# Notebook 01. Entrenamiento comun (todos)
Settings: Accelerator **GPU P100**, Internet **On**. Input: dataset privado **banana-cache-224-v3** (test = días completos que train no ve).
Se cambian solo `MODEL` y `SMOKE_TEST`. Test por días completos, normalización ImageNet para los 6 modelos y sin aumento de datos. Primero se corre con `SMOKE_TEST = True` (1 epoca).

In [ ]:
MODEL = "resnet50"     # cambiar SOLO segun la tabla de asignacion
SMOKE_TEST = True      # True: prueba de 1 epoca | False: entrenamiento completo

TIMM = {"resnet50": "resnet50.a1_in1k",
        "efficientnet_b0": "efficientnet_b0.ra_in1k",
        "mobilenetv2": "mobilenetv2_100.ra_in1k",
        "vit_b16": "vit_base_patch16_224.augreg2_in21k_ft_in1k",
        "swin_t": "swin_tiny_patch4_window7_224.ms_in1k",
        "coatnet_0": "coatnet_0_rw_224.sw_in1k"}
CFG = dict(epochs=50, batch=16, lr=1e-4, wd=1e-4, patience=8, seed=42)
NORM = "imagenet"         # igual para los 6 modelos: media y desviacion de los pesos preentrenados (ImageNet)
USE_CLASS_WEIGHTS = True  # pesos de clase calculados solo con train (stats.json)
if SMOKE_TEST:
    CFG["epochs"] = 1
CLASES = ["Sigatoka", "Cordana", "Healthy"]
assert MODEL in TIMM

In [ ]:
!pip -q install timm==1.0.29
import os, glob, json, time, random, sys
import numpy as np, pandas as pd, torch, torch.nn as nn, timm, sklearn, statsmodels
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from statsmodels.stats.proportion import proportion_confint

DEV = "cuda"
assert torch.cuda.is_available(), "Activa GPU P100 en Settings"
ENV = dict(python=sys.version.split()[0], torch=torch.__version__, timm=timm.__version__,
           sklearn=sklearn.__version__, statsmodels=statsmodels.__version__, gpu=torch.cuda.get_device_name(0))
print(ENV)

def set_seed(s=CFG["seed"]):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def seed_worker(_):
    s = torch.initial_seed() % 2**32; np.random.seed(s); random.seed(s)

CACHE = glob.glob("/kaggle/input/**/X224.npy", recursive=True)
assert len(CACHE) == 1, CACHE
DIR = os.path.dirname(CACHE[0])
meta = pd.read_csv(f"{DIR}/meta.csv")
stats = json.load(open(f"{DIR}/stats.json"))
X = np.load(CACHE[0], mmap_mode="r")
assert "day" in meta.columns, "Estas usando un cache viejo. Usa banana-cache-224-v3"
idx = {s: np.where(meta["split"] == s)[0] for s in ["train", "val", "test"]}
print({s: len(v) for s, v in idx.items()})
assert {s: len(v) for s, v in idx.items()} == {"train": 1886, "val": 236, "test": 232}

In [ ]:
set_seed()
model = timm.create_model(TIMM[MODEL], pretrained=True, num_classes=3).to(DEV)
if NORM == "train":
    MEAN = torch.tensor(stats["mean_train"]).view(3, 1, 1).float()
    STD = torch.tensor(stats["std_train"]).view(3, 1, 1).float()
else:
    dcfg = timm.data.resolve_model_data_config(model)
    MEAN = torch.tensor(dcfg["mean"]).view(3, 1, 1).float()
    STD = torch.tensor(dcfg["std"]).view(3, 1, 1).float()
n_params = sum(p.numel() for p in model.parameters())
print(MODEL, "| parametros:", n_params, "| normalizacion:", NORM, MEAN.flatten().tolist(), STD.flatten().tolist())

class LeafDS(Dataset):
    def __init__(self, split):
        self.ids = idx[split]
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, k):
        i = self.ids[k]
        x = torch.from_numpy(np.array(X[i])).permute(2, 0, 1).float() / 255.0
        x = (x - MEAN) / STD   # sin aumento de datos
        return x.contiguous(), int(meta["label"].iat[i]), meta["sample_id"].iat[i]

def loaders():
    g = torch.Generator(); g.manual_seed(CFG["seed"])
    def mk(s, tr):
        return DataLoader(LeafDS(s), batch_size=CFG["batch"], shuffle=tr, num_workers=2,
                          pin_memory=True, worker_init_fn=seed_worker, generator=g)
    return mk("train", True), mk("val", False), mk("test", False)

dl_tr, dl_va, dl_te = loaders()
W = torch.tensor(stats["pesos_clase_train"], dtype=torch.float32).to(DEV) if USE_CLASS_WEIGHTS else None
print("Imagenes de train por clase:", stats["imagenes_train_por_clase"], "| pesos de clase:", None if W is None else W.cpu().numpy().round(3))

In [ ]:
OUT = f"/kaggle/working/{'prueba_1_epoca' if SMOKE_TEST else 'resultados'}/{MODEL}"
os.makedirs(OUT, exist_ok=True)
crit = nn.CrossEntropyLoss(weight=W)

def run_epoch(dl, opt=None, scaler=None):
    train = opt is not None; model.train(train)
    tot, ys, ps = 0.0, [], []
    for x, y, _ in dl:
        x, y = x.to(DEV, non_blocking=True), y.to(DEV)
        with torch.set_grad_enabled(train), torch.amp.autocast("cuda"):
            out = model(x); loss = crit(out, y)
        if train:
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * len(y); ys += y.tolist(); ps += out.argmax(1).tolist()
    return tot / len(ys), accuracy_score(ys, ps), f1_score(ys, ps, average="macro")

opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=.5, patience=3)
scaler = torch.amp.GradScaler("cuda")
best, wait, hist, t0 = 1e9, 0, [], time.time()
for ep in range(1, CFG["epochs"] + 1):
    tl, ta, tf = run_epoch(dl_tr, opt, scaler)
    vl, va, vf = run_epoch(dl_va)
    sched.step(vl)
    hist.append(dict(epoch=ep, train_loss=tl, val_loss=vl, train_acc=ta, val_acc=va,
                     train_f1=tf, val_f1=vf, lr=opt.param_groups[0]["lr"]))
    print(f"ep {ep:02d} | train {tl:.4f} {ta:.3f} | val {vl:.4f} {va:.3f} f1 {vf:.3f}")
    if vl < best:
        best, wait = vl, 0; torch.save(model.state_dict(), f"{OUT}/best_{MODEL}.pt")
    else:
        wait += 1
    if wait >= CFG["patience"]:
        print("parada temprana"); break
train_min = (time.time() - t0) / 60
pd.DataFrame(hist).to_csv(f"{OUT}/history_{MODEL}.csv", index=False)

In [ ]:
model.load_state_dict(torch.load(f"{OUT}/best_{MODEL}.pt", map_location=DEV)); model.eval()
ys, ps, probs, sids = [], [], [], []
torch.cuda.synchronize(); t1 = time.time()
with torch.no_grad():
    for x, y, sid in dl_te:
        with torch.amp.autocast("cuda"):
            logits = model(x.to(DEV))
        pr = torch.softmax(logits.float(), 1).cpu()
        ys += y.tolist(); ps += pr.argmax(1).tolist(); probs.append(pr); sids += list(sid)
torch.cuda.synchronize()
ms_img = (time.time() - t1) / len(ys) * 1000
probs = torch.cat(probs).numpy()
pred = pd.DataFrame({"sample_id": sids, "y_true": ys, "y_pred": ps})
for k in range(3):
    pred[f"p{k}"] = probs[:, k]
pred.to_csv(f"{OUT}/pred_test_{MODEL}.csv", index=False)

k_ok = int((np.array(ys) == np.array(ps)).sum())
lo, hi = proportion_confint(k_ok, len(ys), alpha=.05, method="wilson")
P, R, F, _ = precision_recall_fscore_support(ys, ps, labels=[0, 1, 2], zero_division=0)
cm = confusion_matrix(ys, ps, labels=[0, 1, 2])
res = dict(model=MODEL, timm_name=TIMM[MODEL], smoke_test=SMOKE_TEST, accuracy=k_ok / len(ys),
           ci95_low=lo, ci95_high=hi, f1_macro=f1_score(ys, ps, average="macro"),
           precision_macro=float(P.mean()), recall_macro=float(R.mean()),
           precision=P.tolist(), recall=R.tolist(), f1=F.tolist(), confusion=cm.tolist(),
           params=n_params, epochs_run=len(hist), train_minutes=train_min,
           inference_ms_per_image=ms_img, cfg=CFG, norm=NORM, class_weights=USE_CLASS_WEIGHTS,
           augmentation=False, split="test por dias (v3)", env=ENV)
json.dump(res, open(f"{OUT}/metrics_{MODEL}.json", "w"), indent=1)
assert len(pred) == 232 and cm.sum() == 232
print({k: v for k, v in res.items() if k not in ("precision", "recall", "f1", "confusion")})

In [ ]:
plt.rcParams.update({"font.family": "serif", "font.size": 11})
h = pd.DataFrame(hist)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(h.epoch, h.train_loss, "k-", label="Training"); ax[0].plot(h.epoch, h.val_loss, "k--", label="Validation")
ax[1].plot(h.epoch, h.train_acc, "k-", label="Training"); ax[1].plot(h.epoch, h.val_acc, "k--", label="Validation")
for a in ax:
    a.set_xlabel("Epoch"); a.legend()
ax[0].set_ylabel("Loss"); ax[1].set_ylabel("Accuracy")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_curves_{MODEL}.pdf"); plt.close()

fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.imshow(cm, cmap="Greys")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks(range(3), CLASES); ax.set_yticks(range(3), CLASES)
ax.set_xlabel("Predicted class"); ax.set_ylabel("Actual class")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_cm_{MODEL}.pdf"); plt.close()
print("PRUEBA DE 1 EPOCA OK" if SMOKE_TEST else "ENTRENAMIENTO COMPLETO OK", "|", OUT)
!ls -la {OUT}